# Scan history

**What has actually been measured, when, and how has the register moved across those
measurements?**

Every other notebook shows one scan. This one shows the run log, which is what makes the
others trustworthy: a metric is only as current as the scan behind it, and disappearance
can only be read as remediation because each scan records which severities it covered.

## How to read this notebook

Every cell answers one question and shows one thing. Run them in order the first time; after
that any cell can be re-run on its own.

**Set the widgets at the top before you run anything.** `catalog` has no default on purpose.
Set the notebook to **Run accessed commands** (the dropdown beside *Run all*) if you want a
widget change to re-run the cells that depend on it — otherwise you will change the filter and
read a chart drawn under the old one.

Everything here reads one scan, pinned in cell 1. Charts that span scans say so in their title.

In [ ]:
import os, sys

_paths = []
try:
    _paths.append(dbutils.widgets.get("module_path"))
except Exception:  # noqa: BLE001 -- the widget does not exist yet on a first run
    pass
_here = os.getcwd()
_paths += [_here, os.path.dirname(_here)]
for _p in _paths:
    if _p and os.path.exists(os.path.join(_p, "panels.py")):
        sys.path.insert(0, _p)
        break
else:
    raise RuntimeError("brick modules are not on sys.path -- see brick/README.md, step 2")

import panels, figures, tiles

PAGE = {"rows": ("25", ["10", "25", "50", "100"])}

panels.declare_widgets(**PAGE)
ctx = panels.context(spark, **{name: str(spec[0]) for name, spec in PAGE.items()})
displayHTML(tiles.scan_zone_from(panels.last_scan(spark, ctx).first()))

All four figures below come off the **ledger**, so they share one population. The
published `OVERALL` row could not: the pipeline computes it once over every severity that
was scanned, and a read-time filter cannot re-derive it — so mixing it in here would put
one filtered number beside three unfiltered ones.

In [ ]:
displayHTML(tiles.history_kpis(panels.all_time(spark, ctx).first()))

## Saved scans

Newest first. The deltas are signed strings rather than coloured cells — the sign carries
the meaning, and a result grid cannot colour a cell anyway.

GAS also shows each run's *mode*, *shape* and *status*. brick's scan log records what a
run covered and what changed, and nothing about how the run itself went, so those three
columns are not here rather than being guessed at.

In [ ]:
display(panels.scan_log(spark, ctx, rows=ctx.int_param('rows', 25)))

## Remediation trends

The full history, scoped to the severity filter above.

In [ ]:
figures.render(
    figures.describe(
        figures.trend(
            panels.trend(spark, ctx, ["open", "resolved"]).toPandas(),
            "scan_ts",
            [
                figures.Series("open", "Open", "#b91c1c", symbol="circle"),
                figures.Series("resolved", "Resolved", "#15803d", dash="6,4",
                               symbol="square"),
            ],
        ),
        "Open and resolved findings at each saved scan.",
    )
)

A Kaplan–Meier median is **not summable across severities**, so this chart shows the
register's own OVERALL series rather than adding up per-severity medians. If you have
narrowed the severity filter, note that this line did not narrow with it — see the note
above the KPI band.

In [ ]:
figures.render(
    figures.describe(
        figures.trend(
            panels.trend(spark, ctx, ["km_median"]).toPandas(),
            "scan_ts",
            [figures.Series("km_median", "Median (KM, all)", figures.ACCENT,
                            fill=True)],
            y_unit="days",
        ),
        "Kaplan-Meier median days to remediation, replayed as of each scan, with still-open findings censored.",
    )
)